Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/betum_tool/models/sam3/notebooks/template_inference.ipynb)

# SAM 3 Zero-Shot Promptable Concept Segmentation on Coffee & Cashew Datasets

**Group 3 — Promptable Concept Segmentation (PCS)**

This notebook implements a zero-shot detection and segmentation pipeline using **SAM 3's Promptable Concept Segmentation (PCS)** capability. We prompt the foundation model with text descriptions of yield/maturity concepts (defined in `common/class_map.json`), convert the output masks to bounding boxes, and evaluate performance with standard COCOEval AP@50.

---

### The Conceptual Challenge

SAM 3's PCS capability takes a **text prompt describing a concept** and returns **segmentation masks with unique identities for all matching instances** in the image.

```
Image + Text Prompt ("unripe cashew nut") 
    → SAM 3 PCS 
    → N instance masks (one per detected object)
    → Derive bounding box from each mask (masks_to_bboxes.py)
    → COCO predictions JSON
    → COCOEval AP@50
```

This zero-shot approach is directly compared against OWL-v2 (Group 1 open-vocabulary detector) and YOLO26 (Group 2 supervised detector) to answer the question: *Does a segmentation foundation model prompted with a concept produce better localization than a detection-specific model?*

## 1. Setup

First, we install the necessary dependencies, clone the repository to access the shared scripts, download the dataset, and format it correctly.

In [ ]:
# @title 1a. Install dependencies
!pip install -q transformers torch torchvision pycocotools Pillow matplotlib numpy gdown


In [ ]:
# @title 1b. Clone the repo.
import os

REPO_DIR = "google-research/betum_tool"

if not os.path.exists("google-research"):
    repo_url = "https://github.com/google-research/google-research.git"
    print("Cloning google-research monorepo (sparse checkout)...")
    # We do a sparse checkout of only the betum_tool directory to avoid downloading 2GB+ of monorepo code.
    !git clone --depth=1 --no-checkout {repo_url} google-research
    !cd google-research && git sparse-checkout set betum_tool && git checkout
    print("✓ Cloned google-research/betum_tool/")
else:
    print("Repo already cloned.")

# Verify common scripts exist
assert os.path.exists(f"{REPO_DIR}/common/yolo_to_coco.py"), "Missing yolo_to_coco.py"
assert os.path.exists(f"{REPO_DIR}/common/class_map.json"), "Missing class_map.json"
print("✓ Common scripts found and updated.")

In [ ]:
# @title 1c. Download dataset from Mendeley
import os
import subprocess
import requests

DATA_DIR = "data"
MENDELEY_URL = "https://data.mendeley.com/public-api/zip/r46c6bpfpf/download/1"
ZIP_NAME = "dataset.zip"

os.makedirs(DATA_DIR, exist_ok=True)

zip_path = os.path.join(DATA_DIR, ZIP_NAME)
if not os.path.exists(zip_path):
    print("Downloading dataset from Mendeley (this may take a few minutes)...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        print("Attempting download with requests...")
        response = requests.get(MENDELEY_URL, headers=headers, stream=True)
        if response.status_code == 200:
            with open(zip_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"Downloaded to {zip_path}")
        else:
            print(f"HTTP Error {response.status_code} with requests.")
            raise Exception(f"HTTP Error {response.status_code}")
            
    except Exception as e:
        print(f"Requests failed: {e}")
        print("Trying fallback with wget...")
        try:
            subprocess.run(["wget", "-U", "Mozilla/5.0", "-O", zip_path, MENDELEY_URL], check=True)
            print(f"Downloaded with wget to {zip_path}")
        except Exception as wget_e:
            print(f"Wget also failed: {wget_e}")
            print("Please manually download the dataset from https://data.mendeley.com/datasets/r46c6bpfpf/1 and upload it to the 'data' folder.")
else:
    print(f"Dataset already downloaded at {zip_path}")

In [ ]:
# @title 1d. Unzip main archive
import zipfile
import os

DATA_DIR = "data"
ZIP_NAME = "dataset.zip"
zip_path = os.path.join(DATA_DIR, ZIP_NAME)

print("Unzipping main archive...")
try:
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print("Done!")
except Exception as e:
    print(f"Error unzipping {zip_path}: {e}")

# Debug: List contents to see what we got
print("Contents of DATA_DIR after unzip:")
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}[{os.path.basename(root)}/]")
    subindent = ' ' * 4 * (level + 1)
    for f in files[:5]: # Show first 5 files to avoid spam
        print(f"{subindent}{f}")
    if len(files) > 5:
        print(f"{subindent}... ({len(files) - 5} more files)")

In [ ]:
# @title 1e. Extract .rar files
import os
import subprocess

DATA_DIR = "data"

# In Google Colab, unrar is usually pre-installed, but we can make sure:
!apt-get install -y unrar

print("Searching for .rar files...")
rar_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(".rar"):
            rar_files.append(os.path.join(root, file))

print(f"Found rar files: {rar_files}")

for rar_path in rar_files:
    print(f"Extracting {rar_path}...")
    try:
        # Extract directly to DATA_DIR so that Cashew and Coffee folders 
        # end up in data/
        subprocess.run(["unrar", "x", "-o+", rar_path, DATA_DIR], check=True)
        print(f"Extracted {rar_path} to {DATA_DIR}")
    except Exception as e:
        print(f"Error extracting {rar_path}: {e}")
        print("Please ensure 'unrar' is installed.")

In [ ]:
# @title 1f. Verify dataset structure
import os

DATA_DIR = "data"

print("Verifying structure...")
for expected in ["Cashew/Cashew-Uganda/images", "Coffee/Batch1/images"]:
    found = False
    for root, dirs, files in os.walk(DATA_DIR):
        if expected in root:
            found = True
            print(f"✓ Found {expected} at {root}")
            break
    if not found:
        print(f"Warning: Could not find expected directory structure containing '{expected}'")
        
print("✓ Verification check complete.")

In [ ]:
# @title 1g. Flatten Coffee + Convert YOLO → COCO JSON
import shutil

# --- Flatten Coffee batches ---
# flatten_coffee.py expects data/Coffee/ in the current directory
coffee_flat_dir = os.path.join(DATA_DIR, "Coffee_flattened")
if os.path.exists(coffee_flat_dir):
    print("Cleaning up existing Coffee_flattened directory to force re-flattening...")
    shutil.rmtree(coffee_flat_dir)

print("Flattening Coffee dataset...")
!python {REPO_DIR}/common/flatten_coffee.py

# --- Convert YOLO → COCO for Cashew ---
os.makedirs(os.path.join(DATA_DIR, "coco"), exist_ok=True)

cashew_coco = os.path.join(DATA_DIR, "coco", "cashew_val.json")
if not os.path.exists(cashew_coco):
    !python {REPO_DIR}/common/yolo_to_coco.py \
        --images {DATA_DIR}/Cashew/Cashew-Uganda/images \
        --labels {DATA_DIR}/Cashew/Cashew-Uganda/Labels \
        --class_map {REPO_DIR}/common/class_map.json \
        --dataset cashew \
        --output {DATA_DIR}/coco/ \
        --split_ratio 0.8
else:
    print("Cashew COCO JSON already exists.")

# --- Convert YOLO → COCO for Coffee ---
coffee_coco = os.path.join(DATA_DIR, "coco", "coffee_val.json")
if not os.path.exists(coffee_coco):
    !python {REPO_DIR}/common/yolo_to_coco.py \
        --images {DATA_DIR}/Coffee_flattened/images \
        --labels {DATA_DIR}/Coffee_flattened/labels \
        --class_map {REPO_DIR}/common/class_map.json \
        --dataset coffee \
        --output {DATA_DIR}/coco/ \
        --split_ratio 0.8
else:
    print("Coffee COCO JSON already exists.")

print("\n✓ COCO ground truth ready.")

### 1h. Visualize Ground Truth Bounding Boxes

Let's use the shared utility `common/visualize_coco.py` to draw ground truth bounding boxes on a few validation images to inspect the annotations before we run inference.

In [ ]:
# @title 1h. Visualize Ground Truth Annotations
import sys
sys.path.append(REPO_DIR)

from common.visualize_coco import visualize

# Visualize 3 random ground truth validation images for Cashew
print("Visualizing cashew validation ground truth annotations:")
visualize(
    coco_json="data/coco/cashew_val.json",
    image_dir="data/Cashew/Cashew-Uganda/images",
    num_images=3,
    output_dir=None
)

### 1i. Hugging Face Hub Authentication (Required)

Because `facebook/sam3` is a gated model repository, you must authenticate with Hugging Face to download the model weights.

1. Create a Hugging Face account if you don't have one.
2. Request access on the [facebook/sam3](https://huggingface.co/facebook/sam3) model page.
3. Generate a **User Access Token** (with Read scope) at: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
4. Run the cell below to authenticate.

In [ ]:
# @title 1i. Log in to Hugging Face Hub
from huggingface_hub import notebook_login

# This will display a text box inside your notebook to enter your HF token
notebook_login()

## 2. Configuration

Adjust these variables for your setup. Set `NUM_EXAMPLES = 4` for a quick smoke test, or `None` for a full run. You can choose either `cashew` or `coffee`.

In [ ]:
# @title Configuration {display-mode: "form"}
from pathlib import Path

# --- Smoke test ---
NUM_EXAMPLES = 4  # Set to None for full dataset

# --- Dataset selection ---
DATASET = "cashew"  # @param ["cashew", "coffee"]

# --- Paths (auto-configured from setup cells above) ---
COCO_VAL_JSON = Path(f"data/coco/{DATASET}_val.json")

IMAGE_DIRS = {
    "cashew": Path("data/Cashew/Cashew-Uganda/images"),
    "coffee": Path("data/Coffee_flattened/images"),
}
IMAGE_DIR = IMAGE_DIRS[DATASET]

# --- Model ---
MODEL_ID = "facebook/sam3"  # Or segment-anything-3 specific model id

# --- Detection ---
CONFIDENCE_THRESHOLD = 0.05  # Start low for zero-shot; tune later
SCORE_THRESHOLDS_TO_SWEEP = [0.01, 0.05, 0.1, 0.15, 0.2]

# --- Text prompts per dataset ---
# Order matches category_ids 0, 1, 2, ... in the COCO JSON.
# ⚡ Experiment with these! Prompt engineering is key for zero-shot.
TEXT_QUERIES = {
    "cashew": [
        "cashew tree",
        "cashew flower",
        "premature cashew nut",
        "unripe cashew nut",
        "ripe cashew nut",
        "spoilt cashew nut",
    ],
    "coffee": [
        "unripe coffee berry",
        "ripening coffee berry",
        "ripe coffee berry",
        "spoilt coffee berry",
        "coffee tree",
    ],
}

print(f"Dataset: {DATASET}")
print(f"NUM_EXAMPLES: {NUM_EXAMPLES}")
print(f"COCO JSON: {COCO_VAL_JSON}")
print(f"Image dir: {IMAGE_DIR}")
print(f"Text queries: {TEXT_QUERIES[DATASET]}")

## 3. Device Selection

Let's select the best available device (CUDA GPU on Colab, MPS on Apple Silicon, or CPU).

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 4. Load Data

Load the COCO ground truth JSON to get the image list and annotations for evaluation. We can subset the images for speed during development.

In [ ]:
import json

# Load COCO ground truth
with open(COCO_VAL_JSON, "r") as f:
    coco_gt_data = json.load(f)

images_info = coco_gt_data["images"]
if NUM_EXAMPLES is not None:
    images_info = images_info[:NUM_EXAMPLES]

print(f"Loaded {len(images_info)} validation images.")

## 5. Load Model

Load SAM 3 from HuggingFace Hub. We will load both the processor/preprocessor and the main model, and move the model to our selected device.

In [ ]:
from transformers import Sam3Model, Sam3Processor

# --- Load Model Configuration ---
print(f"Loading native SAM 3 model {MODEL_ID}...")
processor = Sam3Processor.from_pretrained(MODEL_ID)
model = Sam3Model.from_pretrained(MODEL_ID).to(device)

print(f"\n✓ Native SAM 3 model loaded successfully on {device}!")

## 6. Run Inference and Mask → BBox Conversion

We run inference on all validation images. For each image:
1. Load and preprocess the image.
2. Run SAM 3 Promptable Concept Segmentation (PCS) for each of our text queries.
3. Extract the resulting binary masks.
4. Convert each binary mask to absolute COCO bounding coordinates `[x_min, y_min, width, height]` using our utility `masks_to_bboxes.py`.
5. Collect predictions in standard COCO format.

In [ ]:
import sys
from PIL import Image
import numpy as np
from tqdm import tqdm
import torch

# Append repository directory to access local modules
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

def run_sam3_native_pcs(model, processor, image, prompt_text, device):
    """Run SAM 3 Promptable Concept Segmentation natively from Hugging Face.
    
    Args:
        model: Loaded SAM 3 model.
        processor: SAM 3 processor.
        image: PIL Image object.
        prompt_text: Text prompt describing the concept.
        device: Target torch device.
        
    Returns:
        masks: list of numpy boolean arrays of shape (height, width).
        boxes: list of lists in COCO [x_min, y_min, width, height] format.
        scores: list of float confidence scores.
    """
    # Segment using text prompt
    inputs = processor(images=image, text=prompt_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    # Post-process results
    # We use a low threshold of 0.01 to capture candidates and sweep them later
    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=0.01,
        mask_threshold=0.0,
        target_sizes=inputs.get("original_sizes").tolist()
    )[0]
    
    masks_tensor = results["masks"] # [num_objects, H, W]
    boxes_tensor = results["boxes"] # [num_objects, 4] (xyxy format)
    scores_tensor = results["scores"] # [num_objects]
    
    masks = [m.cpu().numpy() for m in masks_tensor]
    scores = scores_tensor.cpu().tolist()
    
    # Convert xyxy to COCO [x_min, y_min, width, height] format
    boxes = []
    for box in boxes_tensor.cpu().tolist():
        x1, y1, x2, y2 = box
        w = x2 - x1
        h = y2 - y1
        boxes.append([x1, y1, w, h])
        
    return masks, boxes, scores


# --- Core Pipeline Execution ---
print(f"Running native SAM 3 PCS inference on {len(images_info)} images...")
all_predictions = []
queries = TEXT_QUERIES[DATASET]

for img_info in tqdm(images_info):
    img_path = IMAGE_DIR / img_info["file_name"]
    if not img_path.exists():
        img_path = Path("data") / img_info["file_name"]
        
    image = Image.open(img_path).convert("RGB")
    
    for category_id, prompt_text in enumerate(queries):
        # Run native SAM 3 PCS
        masks, boxes, scores = run_sam3_native_pcs(model, processor, image, prompt_text, device)
        
        # Format predictions for COCO JSON
        for box, score in zip(boxes, scores):
            all_predictions.append({
                "image_id": img_info["id"],
                "category_id": category_id,
                "bbox": [int(c) for c in box],
                "score": float(score),
            })

print(f"\n✓ Generated {len(all_predictions)} bounding box predictions using native SAM 3 PCS pipeline.")

## 7. Evaluate with COCOEval

Now we load the predictions into `pycocotools` and evaluate against the ground truth annotations using the standard COCO metric AP@50.

In [ ]:
import sys
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Append repository path to access common modules
if 'REPO_DIR' in globals() and REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

from common.evaluate import evaluate_coco

def run_coco_eval(gt_json_path, predictions, score_threshold=None):
    """Unified wrapper for standardized COCO evaluation."""
    return evaluate_coco(
        gt_json=gt_json_path,
        predictions=predictions,
        score_threshold=score_threshold,
        verbose=True
    )


# Single evaluation at default threshold
print(f"=== Evaluation at threshold {CONFIDENCE_THRESHOLD} ===\n")
results = run_coco_eval(COCO_VAL_JSON, all_predictions, CONFIDENCE_THRESHOLD)
print(f"\nAP@50 = {results['AP@50']}")

## 8. Confidence Threshold Sweep

Sweep confidence thresholds to find the optimal balance between precision and recall. Higher thresholds filter out low-confidence false positives, whereas lower thresholds capture more true instances.

In [ ]:
print("=== Threshold Sweep ===\n")
sweep_results = {}

for thresh in SCORE_THRESHOLDS_TO_SWEEP:
    print(f"\n--- Threshold: {thresh} ---")
    res = run_coco_eval(COCO_VAL_JSON, all_predictions, score_threshold=thresh)
    sweep_results[thresh] = res["AP@50"]

# Print summary table
print("\n=== Threshold Sweep Summary ===")
print("| Threshold | AP@50 |")
print("|-----------|-------|")
for thresh, ap50 in sweep_results.items():
    print(f"| {thresh:<9} | {ap50:<5} |")

## 9. Visualise Predictions

Show predictions vs. ground truth side-by-side on sample images.

**Note:** High confidence threshold (e.g. 0.15) helps keep visualisations clean.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

VIS_THRESHOLD = 0.15  # Threshold for visualisation
NUM_VIS = min(4, len(images_info))  # Number of images to visualise

# Extract categories
categories = {cat["id"]: cat["name"] for cat in coco_gt_data.get("categories", [])}

# Build a colour map for categories
cmap = plt.colormaps.get_cmap("tab10")
cat_colors = {cat_id: cmap(i % 10) for i, cat_id in enumerate(categories.keys())}

# Group GT annotations by image_id
gt_by_image = {}
for ann in coco_gt_data.get("annotations", []):
    img_id = ann["image_id"]
    if img_id not in gt_by_image:
        gt_by_image[img_id] = []
    gt_by_image[img_id].append(ann)

# Group predictions by image_id
pred_by_image = {}
for pred in all_predictions:
    if pred["score"] < VIS_THRESHOLD:
        continue
    img_id = pred["image_id"]
    if img_id not in pred_by_image:
        pred_by_image[img_id] = []
    pred_by_image[img_id].append(pred)


def draw_boxes(ax, boxes_data, is_gt=True):
    """Draw bounding boxes on an axis."""
    for item in boxes_data:
        bbox = item["bbox"]  # [x, y, w, h]
        cat_id = item["category_id"]
        color = cat_colors.get(cat_id, "red")
        linestyle = "-" if is_gt else "--"
        linewidth = 2 if is_gt else 1.5

        rect = patches.Rectangle(
            (bbox[0], bbox[1]), bbox[2], bbox[3],
            linewidth=linewidth, edgecolor=color,
            facecolor="none", linestyle=linestyle,
        )
        ax.add_patch(rect)

        label = categories.get(cat_id, f"cls{cat_id}")
        if not is_gt and "score" in item:
            label = f"{label} {item['score']:.2f}"
        ax.text(
            bbox[0], bbox[1] - 4, label,
            fontsize=7, color=color,
            bbox=dict(boxstyle="round,pad=0.15", facecolor="black", alpha=0.6),
        )


fig, axes = plt.subplots(NUM_VIS, 2, figsize=(16, 5 * NUM_VIS))
if NUM_VIS == 1:
    axes = axes.reshape(1, -1)

for row, img_info in enumerate(images_info[:NUM_VIS]):
    img_path = IMAGE_DIR / img_info["file_name"]
    if not img_path.exists():
        img_path = Path("data") / img_info["file_name"]
    image = Image.open(img_path).convert("RGB")
    img_id = img_info["id"]

    # Ground truth
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"Ground Truth — {img_info['file_name']}", fontsize=10)
    axes[row, 0].axis("off")
    gt_anns = gt_by_image.get(img_id, [])
    draw_boxes(axes[row, 0], gt_anns, is_gt=True)

    # Predictions
    axes[row, 1].imshow(image)
    axes[row, 1].set_title(
        f"SAM 3 PCS Predictions (threshold={VIS_THRESHOLD})", fontsize=10
    )
    axes[row, 1].axis("off")
    pred_anns = pred_by_image.get(img_id, [])
    draw_boxes(axes[row, 1], pred_anns, is_gt=False)

plt.tight_layout()
plt.show()

## 10. Save Predictions

Save the COCO-format predictions to disk. These are valuable for cross-model analysis and comparative evaluations with the other groups in the class!

In [ ]:
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"{DATASET}_predictions.json"
with open(output_path, "w") as f:
    json.dump(all_predictions, f, indent=2)

print(f"✓ Predictions saved to {output_path}")

## 11. Prompt Engineering Experiments

Prompt engineering is a vital component of working with open-vocabulary/segmentation foundation models. In this section, you will systematically vary prompt wordings to see their impact on the final AP@50 metric.

Try to evaluate at least 3 different configurations and record your results in `results/ablations.md`.

### Suggested Ablations:

1. **Simple Class Names:** Just `tree`, `flower`, `unripe`, etc.
2. **Contextual Phrases:** `a photo of a unripe coffee berry`, `dense cashew tree leaves`, etc.
3. **Domain-Specific Descriptions:** Using high-fidelity terms from the dataset paper or class map.

Fill in the table in `results/ablations.md` and discuss the findings during the group presentations!